# Etapa 1 — Auditoria e exploração

FIAP Global Solution 2026.2. Classificação de ocorrências com MLP. Esta etapa não treina modelos nem prevê crimes futuros. Fontes: os três XLSX CelularesSubtraidos_2023/2024/2025, fornecidos em D:/, com metodologia e dicionário próprios. Os originais não são carregados neste notebook. Proveniência completa em `../docs/decisoes.md`.

In [1]:
from pathlib import Path
import csv, json
ROOT = Path.cwd()
if not (ROOT / 'reports').exists(): ROOT = ROOT.parent
assert (ROOT / 'reports/inventario.json').exists(), 'Execute o pipeline antes do notebook'
def tabela(nome):
    with (ROOT / f'reports/tables/{nome}.csv').open(encoding='utf-8-sig', newline='') as f:
        rows=list(csv.reader(f))
    for row in rows: print(' | '.join(row))


## Estrutura e proveniência

55 colunas por ano; 3 abas por XLSX. Em 2025, os campos de mês/ano passam a ser explicitamente do registro. Não os interpretar como data do fato. As tabelas completas de esquema e dicionário ficam em reports/tables.

In [2]:
inventory=json.loads((ROOT/'reports/inventario.json').read_text(encoding='utf-8'))
for f in inventory:
    print(Path(f['path']).name, 'linhas:', f['rows'], 'colunas:',len(f['columns']), 'SHA256:',f['sha256'])


CelularesSubtraidos_2023.xlsx linhas: 490154 colunas: 55 SHA256: ff84477ca21570c0dcdfd0b6f286313c50d73e9ff5e7aace3d48c025e230ed9c
CelularesSubtraidos_2024.xlsx linhas: 412527 colunas: 55 SHA256: 2cd2615921681731e11fb0fa2d26887f6bdf4e44773db0f8d09ca26780bcc459
CelularesSubtraidos_2025.xlsx linhas: 383635 colunas: 55 SHA256: b86cc5576688bff2aa0b65a244c9a396d5e49e9c4eaf05b4e4dd665f1c2956c9


## Geografia e granularidade

Filtro pelo município de circunscrição, com código IBGE corroborativo. Município de registro não define o local do fato. Cada linha original representa um vínculo; a unidade consolidada é a tupla de delegacia, ano e número do BO. Naturezas, datas ou geografia incompatíveis levam à segregação, com rastreabilidade local.

In [3]:
tabela('recorte_linhas')
tabela('granularidade')
tabela('decisoes_boletins')


ano_arquivo | originais | linhas_capital | municipio_indeterminado | outros_municipios | conflitos_ibge | chave_incompleta | hora_invalida
2023 | 490154 | 267899 | 0 | 222255 | 0 | 0 | 0
2024 | 412527 | 223839 | 0 | 188688 | 0 | 0 | 0
2025 | 383635 | 216287 | 0 | 167348 | 0 | 0 | 0
boletins | multiplas_linhas | linhas_acima_de_um_por_bo | em_multiplos_arquivos | multiplas_versoes | multiplas_naturezas | classes_mistas | datas_conflitantes
995565 | 120140 | 290751 | 4 | 12 | 33975 | 267 | 0
decisao | boletins | linhas_origem | pct_boletins
elegivel | 505187 | 566582 | 50.7437
fora_capital_ou_indeterminado | 411798 | 570352 | 41.3632
sem_classe_binaria | 61452 | 65158 | 6.1726
outras_naturezas_associadas | 12708 | 64564 | 1.2765
fato_fora_2023_2025 | 2879 | 3311 | 0.2892
geografia_conflitante | 1392 | 15500 | 0.1398
roubo_e_furto_no_bo | 149 | 849 | 0.015


## Classes e período

0=FURTO (rubrica exata art. 155), 1=ROUBO (rubrica exata art. 157). Outras naturezas e classes conflitantes não são rotuladas por substring. O ano deriva da data do fato. A base não constitui a estatística oficial da SSP.

In [4]:
tabela('classes_total')
tabela('classes_por_ano')
tabela('datas_data_ocorrencia_bo')


alvo | boletins | percentual
0 | 298083 | 59.0045
1 | 207104 | 40.9955
ano_fato | alvo | boletins | pct_ano
2023 | 0 | 106421 | 55.9745
2023 | 1 | 83703 | 44.0255
2024 | 0 | 92055 | 57.9345
2024 | 1 | 66840 | 42.0655
2025 | 0 | 99607 | 63.782
2025 | 1 | 56561 | 36.218
ano_arquivo | menor | maior | invalidas | ausentes | fora_ano_arquivo
2023 | 2003-11-01 | 2025-02-04 | 0 | 0 | 6052
2024 | 2005-02-09 | 2025-01-16 | 0 | 0 | 4863
2025 | 2005-11-25 | 2025-12-31 | 0 | 0 | 5068


## Exploração e qualidade

Há desbalanceamento moderado, mudança da proporção entre anos e ausências elevadas de hora/período. Bairro tem alta cardinalidade e grafias livres. Essas constatações orientam inspeção e engenharia; utilidade preditiva não foi medida. As tabelas de qualidade temporal permitem inspecionar a ausência complementar, sem imputar horas arbitrárias.

In [5]:
attrs=json.loads((ROOT/'reports/perfil_atributos.json').read_text(encoding='utf-8'))
for a in attrs: print(a)
tabela('qualidade_temporal_atributos')


{'atributo': 'mes', 'linhas': 505187, 'ausentes': 0, 'ausentes_pct': 0.0, 'cardinalidade': 12}
{'atributo': 'dia_semana', 'linhas': 505187, 'ausentes': 0, 'ausentes_pct': 0.0, 'cardinalidade': 7}
{'atributo': 'periodo_fato', 'linhas': 505187, 'ausentes': 0, 'ausentes_pct': 0.0, 'cardinalidade': 5}
{'atributo': 'tipo_local', 'linhas': 505187, 'ausentes': 18752, 'ausentes_pct': 3.7119, 'cardinalidade': 28}
{'atributo': 'bairro', 'linhas': 505187, 'ausentes': 1691, 'ausentes_pct': 0.3347, 'cardinalidade': 2328}
ano_fato | alvo | boletins | hora_ausente | periodo_ausente | ambos_ausentes | ambos_presentes | hora_zero
2023 | 0 | 106421 | 78745 | 27676 | 0 | 0 | 381
2023 | 1 | 83703 | 7558 | 76145 | 0 | 0 | 2437
2024 | 0 | 92055 | 70137 | 21918 | 0 | 0 | 313
2024 | 1 | 66840 | 7270 | 59570 | 0 | 0 | 1791
2025 | 0 | 99607 | 74707 | 24900 | 0 | 0 | 375
2025 | 1 | 56561 | 6465 | 50096 | 0 | 0 | 1569


![Painel exploratório](../reports/figures/painel_exploratorio.png)

## Preparação e integridade

A base de modelagem tem apenas cinco candidatos contextuais e alvo. Hora e período brutos foram excluídos como entradas separadas, porque o padrão de preenchimento varia fortemente entre classes. O período harmonizado usa a convenção 00–05 madrugada, 06–11 manhã, 12–17 tarde e 18–23 noite, ou o período declarado na ausência de hora; “Em hora incerta” é INCERTO. As faixas são uma decisão operacional, não uma definição confirmada da SSP. Não existe indicador de qual campo veio preenchido. Chaves, data completa e município estão em metadados alinhados por índice. Não há natureza como entrada, endereço, coordenada precisa ou identificadores administrativos. Nenhum imputador, codificador ou escalonador foi ajustado. Os testes confrontam os resultados com as linhas originais e conferem os hashes.

In [6]:
print((ROOT/'reports/testes_integridade.json').read_text(encoding='utf-8'))
print((ROOT/'data/processed/esquema_modelagem.json').read_text(encoding='utf-8'))


{
  "executados": 8,
  "falhas": 0,
  "erros": 0,
  "sucesso": true
}
{
  "features": [
    "mes",
    "dia_semana",
    "periodo_fato",
    "tipo_local",
    "bairro"
  ],
  "target": "alvo",
  "labels": {
    "0": "FURTO",
    "1": "ROUBO"
  },
  "metadata": "metadados_modelagem.parquet",
  "alignment": "mesma ordenacao por chave_bo; indice_linha zero-based",
  "rows": 505187,
  "sample_rows": 20000,
  "split": "nao realizado",
  "fitted_transformers": false
}


## Cenários e próximos passos

A amostra de 20 mil BOs preserva proporções por ano/mês/classe com cotas de maior resto e semente determinística. A base ampliada melhora cobertura, com maior custo e sem promessa de maior acurácia. Proposta ainda não fixada: 2023 treino, 2024 validação, 2025 teste. Ajustar transformadores e categorias raras somente no treino; manter BOs indivisos.

Revisar com o usuário: exclusão de outras naturezas associadas, bairros de alta cardinalidade, representação conjunta hora/período e a divisão temporal. Ver o relatório completo em `../reports/relatorio_etapa1.md`. O trabalho termina nesta etapa.